# BuoyNet - Quantization Aware Training (QAT) Pipeline
This notebook implements the fine-tuning QAT pipeline described in the paper to optimize the MobileNetV3-Small INT8 model.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torchvision.models import mobilenet_v3_small
from pathlib import Path
import os
import pandas as pd

## 1. Load Data
Using ImageFolder to load the training dataset. We require real data for QAT convergence.

In [ ]:
DATA_DIR = Path('../data')
MODELS_DIR = Path('../models')
LOGS_DIR = Path('../logs')
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(LOGS_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

BATCH_SIZE = 32

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def get_dataloaders():
    train_path = DATA_DIR / 'train'
    EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp'}
    
    if train_path.exists():
        train_count = sum(1 for p in train_path.rglob('*') if p.is_file() and p.suffix.lower() in EXTENSIONS)
        if train_count > 0:
            from PIL import Image, UnidentifiedImageError
            def safe_pil_loader(path):
                try:
                    with open(path, 'rb') as f:
                        img = Image.open(f)
                        return img.convert('RGB')
                except Exception as e:
                    print(f"Warning: Skipping corrupted image: {path}")
                    return Image.new('RGB', (224, 224), (0, 0, 0))
            
            train_ds = ImageFolder(root=str(train_path), transform=train_transform, loader=safe_pil_loader)
            NUM_CLASSES = len(train_ds.classes)
            print(f"Real dataset loaded. Found {NUM_CLASSES} classes.")
            return DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True), NUM_CLASSES
    
    raise FileNotFoundError("Real data not found. QAT requires real data. Check your data path.")

train_loader, NUM_CLASSES = get_dataloaders()

## 2. Load Pre-trained FP32 Baseline

In [ ]:
def load_baseline():
    model = mobilenet_v3_small(weights=None)
    model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, NUM_CLASSES)
    
    baseline_path = MODELS_DIR / 'baseline_fp32.pth'
    if baseline_path.exists():
        model.load_state_dict(torch.load(baseline_path, map_location='cpu'))
        print("Loaded FP32 Baseline.")
    else:
        print("WARNING: baseline_fp32.pth not found. Using untrained weights for demonstration.")
    return model

fp32_model = load_baseline()

## 3. Prepare Model for QAT
Fuse operations (Conv+BN+ReLU) where possible and insert FakeQuantize nodes to simulate INT8 precision during the forward and backward passes. This modifies the gradient behavior through the Straight-Through Estimator (STE).

In [ ]:
def prepare_model_qat(model):
    model.eval() # Must be in eval mode for fusion mapping to work
    
    # Note for MobileNetV3 architectures: 
    # Standard explicit layer-by-layer fusion via torch.ao.quantization.fuse_modules
    # is notoriously difficult on MobileNet due to the inverted residual bottlenecks.
    # A production environment specifically utilizing quantized MobileNets will typically
    # pull from torchvision.models.quantization.mobilenet_vX (which has built-in `fuse_model(is_qat=True)`).
    # 
    # For this script applying fine-tuning QAT onto our existing baseline `mobilenet_v3_small` graph,
    # we will enable the default QNNPACK config and inject FakeQuantize nodes over applicable structural blocks
    # utilizing eagerness evaluation.
    model.qconfig = torch.quantization.get_default_qat_qconfig('qnnpack')
    
    model.train()
    torch.quantization.prepare_qat(model, inplace=True)
    print("Model graph prepared with FakeQuantize nodes for STE gradient propagation.")
    return model

qat_model = prepare_model_qat(fp32_model).to(device)

## 4. QAT Fine-Tuning Loop (15 Epochs)
Fine-tune the FakeQuantized active weights to adapt to INT8 boundaries using the Straight-Through Estimator (STE) loss gradient.

In [ ]:
def fine_tune_qat(model, train_loader, epochs=15):
    print(f"\nStarting {epochs}-epoch QAT Fine-Tuning Phase...")
    # Use AdamW with a reduced learning rate as specified in IEEE paper
    optimizer = optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-5)
    criterion = nn.CrossEntropyLoss()
    
    model.train()
    logs = []
    
    for epoch in range(1, epochs + 1):
        train_loss = 0.0
        correct = 0
        total = 0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
        epoch_loss = train_loss / total
        epoch_acc = correct / total
        
        logs.append({'epoch': epoch, 'qat_train_loss': epoch_loss, 'qat_train_acc': epoch_acc})
        print(f"Epoch [{epoch}/{epochs}] | Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.4f}")
            
    print("QAT Fine-Tuning Converged.")
    pd.DataFrame(logs).to_csv(LOGS_DIR / 'qat_training_logs.csv', index=False)
    return model

tuned_qat_model = fine_tune_qat(qat_model, train_loader, epochs=15)

## 5. Convert and Export Quantized INT8 Weights
Finalize the model by stripping FakeQuantize logic and formally casting the tensors to torch.qint8.

In [ ]:
def export_tuned_model(model):
    model.eval()
    model.cpu()
    # Convert FakeQuantize graph into a physical quantized graph
    quantized_model = torch.quantization.convert(model, inplace=False)
    
    out_path = MODELS_DIR / 'qat_int8_baseline.pth'
    torch.save(quantized_model.state_dict(), out_path)
    print(f"\nSuccessfully exported QAT INT8 model to: {out_path}")
    return quantized_model

final_model = export_tuned_model(tuned_qat_model)